# SQL query from table names - Continued

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('api_key')

## The old Prompt

In [3]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [4]:
# Import necessary libraries
import sqlite3

# Create a connection to an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create Table 1: Employees
cursor.execute('''
CREATE TABLE Employees (
    EmployeeID INTEGER PRIMARY KEY,
    FirstName TEXT NOT NULL,
    LastName TEXT NOT NULL,
    DepartmentID INTEGER,
    Salary REAL
)
''')

# Create Table 2: Departments
cursor.execute('''
CREATE TABLE Departments (
    DepartmentID INTEGER PRIMARY KEY,
    DepartmentName TEXT NOT NULL
)
''')

# Create Table 3: Projects
cursor.execute('''
CREATE TABLE Projects (
    ProjectID INTEGER PRIMARY KEY,
    ProjectName TEXT NOT NULL,
    Budget REAL
)
''')

# Insert sample data into Employees
cursor.executemany('''
INSERT INTO Employees (EmployeeID, FirstName, LastName, DepartmentID, Salary)
VALUES (?, ?, ?, ?, ?)
''', [
    (1, 'Alice', 'Smith', 1, 75000),
    (2, 'Bob', 'Johnson', 2, 80000),
    (3, 'Charlie', 'Williams', 1, 72000)
])

# Insert sample data into Departments
cursor.executemany('''
INSERT INTO Departments (DepartmentID, DepartmentName)
VALUES (?, ?)
''', [
    (1, 'Engineering'),
    (2, 'Marketing')
])

# Insert sample data into Projects
cursor.executemany('''
INSERT INTO Projects (ProjectID, ProjectName, Budget)
VALUES (?, ?, ?)
''', [
    (1, 'Project Alpha', 500000),
    (2, 'Project Beta', 300000)
])

# Commit changes
conn.commit()

# Verify the data
print("Employees:")
for row in cursor.execute('SELECT * FROM Employees'):
    print(row)

print("\nDepartments:")
for row in cursor.execute('SELECT * FROM Departments'):
    print(row)

print("\nProjects:")
for row in cursor.execute('SELECT * FROM Projects'):
    print(row)

# Close the connection
conn.close()

Employees:
(1, 'Alice', 'Smith', 1, 75000.0)
(2, 'Bob', 'Johnson', 2, 80000.0)
(3, 'Charlie', 'Williams', 1, 72000.0)

Departments:
(1, 'Engineering')
(2, 'Marketing')

Projects:
(1, 'Project Alpha', 500000.0)
(2, 'Project Beta', 300000.0)


In [7]:
# FEW SHOT SAMPLES
# Define the context variable as a list if not already defined
context = []

context.append({'role': 'system', 'content': """
-- Maintain the SQL order simple and efficient as you can, using valid SQLite, answer the following questions for the tables provided above.

-- 1. Retrieve the full names of all employees and their department names.
SELECT Employees.FirstName || ' ' || Employees.LastName AS FullName, Departments.DepartmentName
FROM Employees
JOIN Departments ON Employees.DepartmentID = Departments.DepartmentID;

-- 2. Find the total salary paid in each department.
SELECT Departments.DepartmentName, SUM(Employees.Salary) AS TotalSalary
FROM Employees
JOIN Departments ON Employees.DepartmentID = Departments.DepartmentID
GROUP BY Departments.DepartmentName;

-- 3. List all projects with their budgets.
SELECT ProjectName, Budget
FROM Projects;

-- 4. Find the highest-paid employee and their department.
SELECT Employees.FirstName || ' ' || Employees.LastName AS FullName, Departments.DepartmentName, MAX(Employees.Salary) AS HighestSalary
FROM Employees
JOIN Departments ON Employees.DepartmentID = Departments.DepartmentID;

-- 5. Count the number of employees in each department.
SELECT Departments.DepartmentName, COUNT(Employees.EmployeeID) AS EmployeeCount
FROM Employees
JOIN Departments ON Employees.DepartmentID = Departments.DepartmentID
GROUP BY Departments.DepartmentName;
"""})

In [8]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [13]:
#new
context_user = context.copy()
print(return_CCRMSQL("""salary of alice""", context_user))

SELECT Salary
FROM Employees
WHERE FirstName = 'Alice';


In [14]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("salary of alice", old_context_user))

This is your SQL:
```sql
SELECT salary
FROM employees
JOIN salary ON employees.ID_usr = salary.ID_usr
WHERE name = 'Alice';
```

This SQL query retrieves the salary of the employee named Alice by joining the "employees" and "salary" tables on the ID_usr field and filtering the results based on the employee's name.


In [15]:
#new
print(return_CCRMSQL("salary of bob", context_user))

SELECT Salary
FROM Employees
WHERE FirstName = 'Bob';


In [16]:
#old
print(return_CCRMSQL("salary of bob", old_context_user))

This is your SQL:
```sql
SELECT salary
FROM employees
JOIN salary ON employees.ID_usr = salary.ID_usr
WHERE name = 'Bob';
```

This SQL query retrieves the salary of the employee named Bob by joining the "employees" and "salary" tables on the ID_usr column and filtering the results to only include Bob's salary.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

in this exercised i learned that the new prompt is more effective in generating accurate SQL queries from natural language descriptions. The structured format of the table definitions and the inclusion of sample rows provide clearer context for the model, leading to fewer hallucinations and more precise outputs. yet i find the whole task of generating sql prompts by training a model completely useless as it is just a matter of training the model to understand the structure of SQL and the specific requirements of the queries. since u can just feed a database into a model like GPT why would u do the afford of training a model urself to do it? in any real life work scenario u would just put the database of the company into the model and ask it to generate the SQL queries based on whats needed.